# Assignment 1 development notebook

This notebook is a working copy of the initial EC2 and FastAPI prototype.
The provisioning cell creates nine new EC2 instances each time it runs.
Do not use Run All until provisioning and cleanup are made repeatable.
Configure AWS credentials and the SSH key for your own environment before deploying.


In [ ]:
%pip install boto3 paramiko aiohttp

In [ ]:
import boto3, os, paramiko, requests, random, asyncio, aiohttp, time, hashlib

In [ ]:
# Compute team seed
student_ids = sorted(["1879356", "2554694", "2555144"]) # your team's IDs, as strings
joined = "-".join(student_ids)
TeamSeed = int(hashlib.sha256(joined.encode()).hexdigest(), 16) % 10000
print(f'Team seed is {TeamSeed}')

In [ ]:
# Local private key pair file
key_file = r"E:\AWS\labsuser.pem"
key_name = "vockey"

In [ ]:
# constant
region_name="us-east-1"

N_small_instance = 5
small_cluster = 'cluster1'
smallInstanceType="t3.micro"
small_ImageId="ami-0fef201115eefe936"

# c7g.large is cheaper than m7g.large
N_large_instance = 4
large_cluster = 'cluster2'
largeInstanceType="c7g.large"
large_ImageId="ami-0eb45f74aa8a20238"

# create ec2 object
# boto3.client = API-oriented, more complete (specify the operation first, then pass the resource as a parameter)
# boto3.resource = object-oriented, convenient for working with a specific resource (specify the resource first, then perform an operation on that resource)
ec2_cli = boto3.client("ec2", region_name=region_name)
ec2_res = boto3.resource("ec2", region_name=region_name)
elbv2_cli = boto3.client("elbv2", region_name=region_name)

response = ec2_cli.describe_vpcs(
    Filters=[
        {
            "Name": "is-default",
            "Values": ["true"]
        }
    ]
)

VpcId = response["Vpcs"][0]["VpcId"]

In [ ]:
# Select public default subnets from the VPC used by this lab.
def select_public_default_subnets(ec2_client, vpc_id):
    paginator = ec2_client.get_paginator("describe_subnets")
    subnets = {}
    for page in paginator.paginate(Filters=[{"Name": "vpc-id", "Values": [vpc_id]}]):
        for subnet in page["Subnets"]:
            if (subnet["VpcId"] != vpc_id
                    or subnet["State"] != "available"
                    or not subnet["DefaultForAz"]
                    or not subnet["MapPublicIpOnLaunch"]):
                continue
            zone = subnet["AvailabilityZone"]
            if zone in subnets and subnets[zone] != subnet["SubnetId"]:
                raise RuntimeError(f"Multiple default subnets found in {zone}")
            subnets[zone] = subnet["SubnetId"]
    if len(subnets) < 2:
        raise RuntimeError("Need at least two public default subnets in different availability zones")
    return dict(sorted(subnets.items()))

subnet_dict = select_public_default_subnets(ec2_cli, VpcId)
print(subnet_dict)

# Clusters setup

In [ ]:
# Update the OS packages and install Python3 when deploying the EC2 instance
user_data = """#!/bin/bash
sudo dnf update -y
sudo dnf install python3 python3-pip -y
"""

def find_existing_lab_instance_ids(ec2_client, instance_names, vpc_id):
    # Stop before creating resources when this lab already has EC2 instances.
    filters = [
        {"Name": "tag:Name", "Values": instance_names},
        {"Name": "vpc-id", "Values": [vpc_id]},
        {"Name": "instance-state-name", "Values": ["pending", "running", "stopping", "stopped", "shutting-down"]},
    ]
    paginator = ec2_client.get_paginator("describe_instances")
    return sorted({
        instance["InstanceId"]
        for page in paginator.paginate(Filters=filters)
        for reservation in page["Reservations"]
        for instance in reservation["Instances"]
    })

def create_ec2_instances(ImageId,
                         InstanceType,
                         selected_subnet,
                         SecurityGroupIds,
                         KeyName,
                         instance_name,
                        countMinMax = [1,1],
                        user_data = user_data
                        ):
    
    return ec2_res.create_instances(
        ImageId=ImageId,
        InstanceType=InstanceType,
        MinCount=countMinMax[0],
        MaxCount=countMinMax[1],
        KeyName=KeyName,
        SubnetId=subnet_dict[selected_subnet],
        SecurityGroupIds=SecurityGroupIds,
        UserData=user_data,
        TagSpecifications=[{
            "ResourceType": "instance",
            "Tags": [
                {"Key": "Name", "Value": instance_name},
                {"Key": "TeamSeed", "Value": str(TeamSeed)},
            ],
        }]
    )

def create_ec2_tag(InstanceId, key, value):
    return ec2_cli.create_tags(
        Resources=[InstanceId],
        Tags=[
            {
                "Key": key,
                "Value": value
            }
        ]
    )

def create_security_gp(gpname, gpdescription, inboundport):
    # Security Group exists?
    response = ec2_cli.describe_security_groups(
    Filters=[
        {
            "Name": "group-name",
            "Values": [gpname]
        },
        {
            "Name": "vpc-id",
            "Values": [VpcId]
        }
    ]
    )
    
    # Get the Security Group ID if it exists
    if response["SecurityGroups"]:
        security_group_id = response["SecurityGroups"][0]["GroupId"]

        # Remove all existing inbound rules (Who is allowed to access my EC2 instance?)
        if response["SecurityGroups"][0]["IpPermissions"]:
            ec2_cli.revoke_security_group_ingress(
                GroupId=security_group_id,
                IpPermissions=response["SecurityGroups"][0]["IpPermissions"]
            )
            
    # Create a new Security Group if it does not exist.
    # https://docs.aws.amazon.com/boto3/latest/reference/services/ec2/client/create_security_group.html
    else:
        response = ec2_cli.create_security_group(
        GroupName=gpname,
        Description=gpdescription,
        VpcId=VpcId
        )
        security_group_id = response["GroupId"]
    
    # Adds the specified inbound (ingress) rules to a security group.
    # https://docs.aws.amazon.com/boto3/latest/reference/services/ec2/client/authorize_security_group_ingress.html
    for i in inboundport:
        ec2_cli.authorize_security_group_ingress(
        GroupId=security_group_id,
        IpPermissions=[
            {
                "IpProtocol": "tcp",
                "FromPort": i,
                "ToPort": i,
                "IpRanges": [
                    {
                        "CidrIp": "0.0.0.0/0" #Allow access to this port from any IPv4 address.
                    }
                ]
            }
        ]
        )
    return security_group_id

In [ ]:
expected_instance_names = (
    [f"small_instance_{i}" for i in range(N_small_instance)]
    + [f"large_instance_{i}" for i in range(N_large_instance)]
)
existing_instance_ids = find_existing_lab_instance_ids(ec2_cli, expected_instance_names, VpcId)
if existing_instance_ids:
    raise RuntimeError(
        "EC2 instances with lab names already exist: " + ", ".join(existing_instance_ids)
        + ". Inspect them before provisioning again."
    )

small_instances, large_instances = [], []

# Security group for ecs
# Port 8000: Web application traffic
# Port 22: SSH access
ec_sg_id = create_security_gp(gpname = 'TP1_sg', gpdescription = 'security group for small + large instances ecs', inboundport = [8000, 22])

# ******************************************************************** 
# cluster small
for i in range(N_small_instance):
    # Randomly select a subnet for each instance
    subnet = list(subnet_dict.keys())[random.randint(0, len(subnet_dict)-1)]

    # create an EC2 instance in that subnet
    small_instances.append(
        create_ec2_instances(ImageId=small_ImageId,
                         InstanceType=smallInstanceType,
                         selected_subnet=subnet,
                         SecurityGroupIds=[ec_sg_id],
                         KeyName=key_name,
                         instance_name=f"small_instance_{i}")[0]
    )
# ******************************************************************** 
# cluster large
for i in range(N_large_instance):
    # Randomly select a subnet for each instance
    subnet = list(subnet_dict.keys())[random.randint(0, len(subnet_dict)-1)]

    # create an EC2 instance in that subnet
    large_instances.append(
        create_ec2_instances(ImageId=large_ImageId,
                         InstanceType=largeInstanceType,
                         selected_subnet=subnet,
                         SecurityGroupIds=[ec_sg_id],
                         KeyName=key_name,
                         instance_name=f"large_instance_{i}")[0]
    )

# All instance IDs
all_instances = small_instances + large_instances
all_instance_ids = [instance.id for instance in all_instances]

# Wait until all instances are in the "running" state
ec2_waiter = ec2_cli.get_waiter("instance_running")

ec2_waiter.wait(
    InstanceIds=all_instance_ids
)

print("All EC2 instances are running")

In [ ]:
# Refresh instance information from AWS
for instance in all_instances:
    instance.reload()

small_instances_ids = [i.id for i in small_instances]
print('Small Ids : ')
print(small_instances_ids)
print('----')

large_instances_ids = [i.id for i in large_instances]
print('Large Ids : ')
print(large_instances_ids)
print('----')

small_instances_ips = [i.public_ip_address for i in small_instances]
print('Small IPs : ')
print(small_instances_ips)
print('----')

large_instances_ips = [i.public_ip_address for i in large_instances]
print('Large IPs : ')
print(large_instances_ips)
print('----')

In [ ]:
# Assign a Name tag to each EC2 instance
small_instances_name = [f'small_instance_{i}' for i in range(N_small_instance)]
for id_, val in zip(small_instances_ids, small_instances_name):
    create_ec2_tag(id_, 'Name', val)

large_instances_name = [f'large_instance_{i}' for i in range(N_large_instance)]
for id_, val in zip(large_instances_ids, large_instances_name):
    create_ec2_tag(id_, 'Name', val)

# FastAPI Deployment

#### 1. Update the OS and install Python (done when deploying the instances)

#### 2. Create the FastAPI `.py` file and deployment `.sh` script
    - .py file defines the FastAPI application
    - .sh script creates the virtual environment, installs dependencies, and starts the application

#### 3. Transfer the `.py` and `.sh` files to EC2

#### 4. Execute the deployment script

In [ ]:
# Generate a FastAPI Python file for one EC2 instance
def generate_fastapi_py(
    cluster,
    instance_name,
    instance_id,
    instance_ip,
    TeamSeed=TeamSeed
):
# Refer to the FastAPI starter template shared on Moodle
    app_code = f"""
from fastapi import FastAPI, Response
import uvicorn
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

app = FastAPI()

@app.get("/{cluster}")
def handle_request(response: Response):
    message = "{instance_name} is responding now ... "

    logger.info(message)

    response.headers["X-Team-Seed"] = "{TeamSeed}"

    return {{
        "message": message,
        "instance_id": "{instance_id}",
        "ip": "{instance_ip}",
        "TeamSeed": "{TeamSeed}"
    }}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)
"""

    # Create the Python file locally and write the FastAPI code into it
    file_name = f"{instance_name}.py"
    with open(file_name, "w", encoding="utf-8", newline="\n") as f:
        f.write(app_code)
    return os.path.abspath(file_name)
# -------------------------------------------------------------------
# Generate a .sh provisioning file for one EC2 instance
def generate_provison_bash(instance_name):
    bash_code = f"""#!/bin/bash
cd /home/ec2-user
python3 -m venv .venv
source .venv/bin/activate
pip install fastapi "uvicorn[standard]"
nohup .venv/bin/uvicorn {instance_name}:app \
    --host 0.0.0.0 \
    --port 8000 \
    > /home/ec2-user/app.log 2>&1 < /dev/null & 
""" 
    # Create the provisioning file locally and write the Linux commands into it
    file_name = f"{instance_name}.sh"
    with open(file_name, "w", encoding="utf-8", newline="\n") as f:
        f.write(bash_code)
    return os.path.abspath(file_name)
# -------------------------------------------------------------------
# Transfer the .py and .sh files to EC2 instance & Execute the deployment script
def transfer_execute_app_sh_file(N_instance, instances_ips, app_address, bash_address, instances_name, cluster, test_app = True):
    for i in range(N_instance):
        ip = instances_ips[i]
        app_name = instances_name[i]
        local_file = app_address[i]
        local_bash_file = bash_address[i]
        remote_file = f"/home/ec2-user/{app_name}.py"
        remote_bash_file = f"/home/ec2-user/{app_name}.sh"

        print(f'{ip} : {app_name}')
    # ---------------------------------------------------------- 
        # Create an SSH client
        ssh = paramiko.SSHClient()
        # Automatically accept the EC2 host key if it is not known yet
        ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
    
        ssh.connect(
            hostname=ip,
            username="ec2-user",
            key_filename=key_file,
        )
        
        # Transfer files using SFTP 
        sftp = ssh.open_sftp()
        sftp.put(local_file, remote_file)
        sftp.put(local_bash_file, remote_bash_file)
    # ---------------------------------------------------------- 
        # Kill any process currently using port 8000
        stdin, stdout, stderr = ssh.exec_command("sudo fuser -k 8000/tcp") 
        # Make the script executable, and then run it
        stdin, stdout, stderr = ssh.exec_command(
            f"chmod +x {app_name}.sh && ./{app_name}.sh"
        )

        # Wait until the provisioning script finishes before closing the SSH connection
        # Otherwise, the deployment may be interrupted
        stdout.channel.recv_exit_status()
    
        ssh.close()
        sftp.close()
    # ---------------------------------------------------------- 
        # Test  
        if test_app : print("response:", requests.get(f"http://{ip}:8000/{cluster}").text, '\n')

In [ ]:
small_app_address, small_bash_address = [], []
large_app_address, large_bash_address = [], []

# small cluster
for ins_name, ins_id, ins_ip in zip(small_instances_name, small_instances_ids, small_instances_ips):
    small_app_address.append(
        generate_fastapi_py(cluster=small_cluster, instance_name=ins_name, instance_id=ins_id, instance_ip=ins_ip)
    )
    small_bash_address.append(
        generate_provison_bash(ins_name)
    )

transfer_execute_app_sh_file(N_small_instance, small_instances_ips, small_app_address, small_bash_address, small_instances_name, small_cluster)

# ***********************************************
# large cluster
for ins_name, ins_id, ins_ip in zip(large_instances_name, large_instances_ids, large_instances_ips):
    large_app_address.append(
        generate_fastapi_py(cluster=large_cluster, instance_name=ins_name, instance_id=ins_id, instance_ip=ins_ip)
    )
    large_bash_address.append(
        generate_provison_bash(ins_name)
    )

transfer_execute_app_sh_file(N_large_instance, large_instances_ips, large_app_address, large_bash_address, large_instances_name, large_cluster)

# Terminate all

In [ ]:
ec2_cli.terminate_instances(InstanceIds = small_instances_ids+large_instances_ids)
# elbv2_cli.delete_load_balancer(LoadBalancerArn=alb_arn)